# Construction du dataset consolidé

Application des décisions prises aux cinq notebooks précédents :

| Décision | Origine | Motif |
|---|---|---|
| Supprimer les 231 rendements négatifs | notebook 01 | physiquement impossibles ; tous sans engrais ni irrigation |
| Conserver `Days_to_Harvest` | notebook 01 | écarter une variable relève de l'étape 2, pas de la préparation |
| Aucune réduction de dimension | notebook 02 | l'ACP ne compresse rien (5 axes pour 90 %) |
| Reconstruire le jeu historique | notebooks 03-04 | `yield_df.csv` : 54 % de redondance, 101 pays seulement |
| Jeu simulé comme socle + profil par culture | notebook 05 | seule clé de jointure possible |
| Pluie du jeu historique en percentile | notebook 05 | saison culturale contre normale annuelle |
| Orge et coton imputés et signalés | notebook 05 | absents du jeu historique |

Tout est implémenté dans `src/data/build_dataset.py`, afin que l'étape 2 et l'API de
l'étape 3 rejouent exactement les mêmes transformations.

# Imports

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))

from src import config, utils
from src.data import build_dataset, crop_yield_prediction as cyp
from src.data import preparation

utils.style()
np.random.seed(config.SEED)

# Préparation du jeu simulé

In [2]:
agri, rapport = preparation.nettoie(preparation.charge())
print(rapport)

               etape                                                       detail
              typage          Fertilizer_Used, Irrigation_Used : texte -> booleen
    doublons stricts 0 — conserves le cas echeant (parcelles simulees distinctes)
 rendements negatifs   231 lignes supprimees (0.0231%) — physiquement impossibles
variables conservees           Days_to_Harvest gardee (choix differe a l'etape 2)
  valeurs manquantes                                                   0 au total
              lignes                                         1,000,000 -> 999,769


# Reconstruction du jeu historique

In [3]:
histo, _ = cyp.construit(verbeux=False)
print(f"{len(histo):,} lignes x {histo.shape[1]} colonnes, "
      f"{histo['iso3'].nunique()} pays, {histo['crop'].nunique()} cultures")

25,527 lignes x 8 colonnes, 203 pays, 10 cultures


# Profils agronomiques par culture

In [4]:
profils_fao = build_dataset.profils_fao(histo)
profils_fao.round(2)

,fao_item,crop_ref_yield_t_ha,crop_ref_yield_p25,crop_ref_yield_p75,crop_pesticides_intensity,crop_temp_niche,crop_rain_pctl,n_observations
0,Potatoes,15.10,10.20,22.57,1670.00,18.13,37.56,3637
1,Cassava,9.38,5.56,12.83,378.34,25.67,69.94,2439
2,Plantains and others,7.85,5.02,11.62,839.54,26.32,76.93,1197
3,Yams,7.82,5.01,11.04,196.00,26.42,68.22,1410
4,Sweet potatoes,7.43,4.27,12.36,949.06,24.41,61.00,2740
5,"Rice, paddy",3.03,2.00,4.48,1745.98,22.28,53.42,2787
6,Wheat,2.31,1.47,3.60,2532.00,16.29,30.23,2866
7,Maize,2.30,1.40,4.62,1573.30,20.21,44.38,3868
8,Soybeans,1.46,0.92,2.04,3649.29,18.82,47.33,2127
9,Sorghum,1.24,0.77,2.50,1670.00,20.88,38.94,2456


**Note sur la définition des descripteurs climatiques.** Ancrer la « température optimale »
d'une culture sur ses pays-années les plus productifs paraissait naturel, mais cette mesure
s'est révélée fausse : elle donnait **12,9 °C pour le blé** et 14,1 °C pour le maïs. En
cause, un biais de confusion — les rendements les plus élevés au monde sont obtenus dans
les pays développés et tempérés, de sorte que l'indicateur mesurait le **niveau de richesse
des pays**, pas l'agronomie.

La **température médiane de culture** a été retenue à la place : elle décrit la niche
thermique réelle de l'espèce et rétablit un ordre conforme aux connaissances agronomiques.

**Une seconde réserve subsiste, et elle n'est pas corrigeable ici.** Cette niche est une
moyenne **annuelle** de pays, alors que la température du jeu simulé porte sur la **saison
culturale** — le notebook 05 le démontre à partir du plancher de 15 °C. L'écart
`temp_dev_from_niche` retranche donc deux grandeurs qui ne couvrent pas la même fenêtre :
il porte l'intention agronomique dont la recommandation a besoin, mais ce n'est **pas** un
écart agronomique mesuré. Il est conservé comme heuristique assumée, à valider ou écarter
à l'étape 2.

In [5]:
profils = build_dataset.profils_par_culture(histo)
build_dataset.controle_agronomique(profils)
print("contrôles agronomiques : ordres respectés\n")
profils.round(2)

contrôles agronomiques : ordres respectés



,Crop,crop_ref_yield_t_ha,crop_ref_yield_p25,crop_ref_yield_p75,crop_pesticides_intensity,crop_temp_niche,crop_rain_pctl,crop_ref_source,donor_item
0,Maize,2.30,1.40,4.62,1573.30,20.21,44.38,FAO,NaN
1,Wheat,2.31,1.47,3.60,2532.00,16.29,30.23,FAO,NaN
2,Rice,3.03,2.00,4.48,1745.98,22.28,53.42,FAO,NaN
3,Soybean,1.46,0.92,2.04,3649.29,18.82,47.33,FAO,NaN
4,Barley,2.31,1.47,3.60,2532.00,16.29,30.23,imputed,Wheat
5,Cotton,2.20,1.34,3.06,1670.00,20.88,38.94,imputed,Sorghum


**Observations :** les profils sont **agronomiquement cohérents**, ce qui est le meilleur
contrôle disponible sur une chaîne de transformation de cette longueur :

- niches thermiques : blé et orge 16,3 °C < soja 18,8 < maïs 20,2 < coton 20,9 < riz 22,3 ;
- niveaux de pluie : blé et orge les plus secs (percentile 30), riz le plus humide (54) ;
- rendements de référence : riz 3,03 > blé 2,31 ≈ maïs 2,30 > coton 2,20 > soja 1,46.

Ces trois ordres sont vérifiés par `controle_agronomique`, qui fait échouer la
construction en cas de régression silencieuse.

# Fusion

In [6]:
consolide, rapport = build_dataset.fusionne(agri, profils)
print(rapport)
print(f"\nconsolidé : {len(consolide):,} lignes x {consolide.shape[1]} colonnes")
consolide.head(3)

           etape                                              detail
        jointure            999,769 lignes conservees, aucun fan-out
      completude aucune valeur manquante dans les colonnes de profil
variable derivee         temp_dev_from_niche = |temperature - niche|
      provenance                    FAO : 666,494, imputed : 333,275

consolidé : 999,769 lignes x 18 colonnes


,Region,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Weather_Condition,Days_to_Harvest,Yield_tons_per_hectare,crop_ref_yield_t_ha,crop_ref_yield_p25,crop_ref_yield_p75,crop_pesticides_intensity,crop_temp_niche,crop_rain_pctl,crop_ref_source,temp_dev_from_niche
0,West,Sandy,Cotton,897.077239,27.676966,False,True,Cloudy,122,6.555816,2.20000,1.335688,3.064313,1670.00,20.880,38.937157,imputed,6.796966
1,South,Clay,Rice,992.673282,18.026142,True,True,Rainy,140,8.527341,3.03120,1.995700,4.482600,1745.98,22.275,53.420498,FAO,4.248858
2,North,Loam,Barley,147.998025,29.794042,False,False,Sunny,106,1.127443,2.30725,1.474350,3.601300,2532.00,16.295,30.231970,imputed,13.499042


# Contrôles du dataset consolidé

In [7]:
ajoutees = [c for c in profils.columns if c not in ("Crop", "donor_item")]
controles = {
    "nombre de lignes préservé": len(consolide) == len(agri),
    "aucune valeur manquante": consolide.isna().sum().sum() == 0,
    "aucun manquant dans les profils": consolide[ajoutees].isna().sum().sum() == 0,
    "6 cultures présentes": consolide["Crop"].nunique() == 6,
    "aucun rendement négatif": (consolide["Yield_tons_per_hectare"] >= 0).all(),
    "fusion idempotente": build_dataset.fusionne(agri, profils)[0].equals(consolide),
}
for libelle, ok in controles.items():
    print(f"[{'OK ' if ok else 'ÉCHEC'}] {libelle}")
assert all(controles.values()), "les contrôles post-fusion ont échoué"

print("\nprovenance de l'enrichissement :")
print(consolide.groupby("crop_ref_source", observed=True).size().to_frame("lignes").to_string())

[OK ] nombre de lignes préservé
[OK ] aucune valeur manquante
[OK ] aucun manquant dans les profils
[OK ] 6 cultures présentes
[OK ] aucun rendement négatif
[OK ] fusion idempotente

provenance de l'enrichissement :
                 lignes
crop_ref_source        
FAO              666494
imputed          333275


**Observations :** deux tiers des lignes portent un enrichissement issu des données FAO,
un tiers (orge et coton) un profil imputé, explicitement signalé. La fusion est
idempotente et n'introduit aucune valeur manquante.

# Export

In [8]:
config.DATA_TRAITE.mkdir(parents=True, exist_ok=True)
config.RAPPORTS.mkdir(parents=True, exist_ok=True)

consolide.to_csv(config.DATASET_CONSOLIDE_CSV, index=False)
build_dataset.ecrit_dictionnaire()

taille = config.DATASET_CONSOLIDE_CSV.stat().st_size / 1024**2
print(f"écrit {config.DATASET_CONSOLIDE_CSV.relative_to(config.RACINE)} "
      f"({len(consolide):,} lignes x {consolide.shape[1]} colonnes, {taille:.1f} Mo)")
print(f"écrit {config.DICTIONNAIRE_DONNEES.relative_to(config.RACINE)}")

pd.DataFrame(build_dataset.DICTIONNAIRE,
             columns=["colonne", "type", "unité", "origine", "description"])

écrit data/processed/crop_dataset_consolidated.csv (999,769 lignes x 18 colonnes, 173.2 Mo)
écrit reports/data_dictionary.md


,colonne,type,unité,origine,description
0,Region,category,-,A,"Zone abstraite (North/South/East/West), pas un..."
1,Soil_Type,category,-,A,"Texture du sol : Sandy, Loam, Chalky, Silt, Cl..."
2,Crop,category,-,A,Culture de la parcelle (6 especes)
3,Rainfall_mm,float64,mm,A,Pluie de saison culturale (100-1000)
4,Temperature_Celsius,float64,degC,A,Temperature moyenne de la parcelle sur la sais...
5,Fertilizer_Used,bool,-,A,Fertilisation appliquee
6,Irrigation_Used,bool,-,A,Parcelle irriguee
7,Weather_Condition,category,-,A,"Sunny, Rainy ou Cloudy — independante de la pl..."
8,Days_to_Harvest,int16,jours,A,Duree du cycle (60-150) ; sans lien avec le re...
9,Yield_tons_per_hectare,float64,t/ha,A,CIBLE — rendement observe


In [9]:
# Read-back: the artifact must be self-contained for step 2.
relu = pd.read_csv(config.DATASET_CONSOLIDE_CSV)
print(f"relu : {relu.shape[0]:,} lignes x {relu.shape[1]} colonnes")
print(f"valeurs manquantes : {relu.isna().sum().sum()}")
print(f"cultures : {sorted(relu['Crop'].unique())}")
relu.describe().T.round(3)

relu : 999,769 lignes x 18 colonnes
valeurs manquantes : 0
cultures : ['Barley', 'Cotton', 'Maize', 'Rice', 'Soybean', 'Wheat']


,count,mean,std,min,25%,50%,75%,max
Rainfall_mm,999769.0,550.078,259.804,100.001,325.063,550.229,774.798,999.998
Temperature_Celsius,999769.0,27.506,7.220,15.000,21.256,27.510,33.754,40.000
Days_to_Harvest,999769.0,104.495,25.953,60.000,82.000,104.000,127.000,149.000
Yield_tons_per_hectare,999769.0,4.651,1.695,0.000,3.419,4.652,5.880,9.963
crop_ref_yield_t_ha,999769.0,2.268,0.454,1.463,2.200,2.307,2.307,3.031
crop_ref_yield_p25,999769.0,1.434,0.315,0.919,1.336,1.474,1.474,1.996
crop_ref_yield_p75,999769.0,3.568,0.868,2.039,3.064,3.601,4.483,4.616
crop_pesticides_intensity,999769.0,2283.232,725.215,1573.305,1670.000,1745.980,2532.000,3649.290
crop_temp_niche,999769.0,19.129,2.247,16.295,16.295,20.207,20.880,22.275
crop_rain_pctl,999769.0,40.755,8.581,30.232,30.232,38.937,47.334,53.420


# Conclusion

Le dataset consolidé est en place : **999 769 lignes et 18 colonnes**, sans aucune valeur
manquante. C'est la source de vérité unique des étapes suivantes, accompagnée de son
dictionnaire des données — lequel servira de base aux schémas Pydantic de l'API à
l'étape 3.

## Ce qui a été fait
- Les décisions différées par l'EDA ont été appliquées et journalisées.
- Le jeu historique a été reconstruit depuis ses sources brutes, portant la couverture de
  101 à 120 pays et supprimant la cause des doublons.
- Chaque parcelle porte désormais le profil agronomique mondial de sa culture.
- Trois contrôles agronomiques protègent la chaîne contre une régression silencieuse.

## Les limites à énoncer dans le rapport métier
1. **Le jeu simulé n'est pas mesuré.** Distributions uniformes, modalités parfaitement
   équilibrées, aucun manquant, quelques rendements négatifs. Le modèle apprendra un monde
   plausible mais synthétique : les prédictions sont des ordres de grandeur.
2. **Un tiers des lignes repose sur une imputation** (orge, coton), signalée mais réelle.
3. **L'enrichissement ne prédit pas la cible** (notebook 05). Le classement de
   `/recommend` sera un post-traitement explicite sur les données FAO.
4. **Aucune donnée économique.** Le service parle de rendement attendu, jamais de
   rentabilité.

## Points reportés aux étapes suivantes
- Trancher le retrait de `Days_to_Harvest` et des variables nominales sans effet.
- Vérifier par l'importance des variables que le modèle n'utilise pas
  `crop_ref_yield_t_ha` comme simple ré-encodage de `Crop`.
- Confirmer l'apport de `temp_dev_from_niche`, que le jeu historique ne valide pas seul.
- **Le CSV pèse 173 Mo, au-delà de la limite de 100 Mo par fichier de GitHub** : à
  trancher avant la CI/CD de l'étape 5 (échantillon stratifié, `.csv.gz` ou Git LFS).